# GPU fine-tuning: Pegasus on SAMSum

This notebook fine-tunes the same base model as the local project (`google/pegasus-cnn_dailymail`) on `knkarthick/samsum`.

1. Upload this notebook to [Google Colab](https://colab.research.google.com/).
2. Choose **Runtime > Change runtime type > T4 GPU** (or another GPU), then save.
3. Run cells from top to bottom. The first run downloads the model and SAMSum dataset.
4. Authorize Google Drive mounting when prompted. The notebook saves checkpoints there so training can resume after a disconnect.

I cannot access or start a runtime in your Google account. Full three-epoch fine-tuning can take hours depending on the assigned GPU; free Colab may disconnect.

In [ ]:
# Install the model, dataset, training, tokenizer, and local ROUGE dependencies.
%pip install -q "transformers>=4.46,<5" "datasets>=2.18,<4" "accelerate>=0.26" "sentencepiece>=0.2,<1" "protobuf>=4.25,<7" "rouge-score>=0.1.2,<1"

In [ ]:
# Import PyTorch so we can check whether Colab assigned a CUDA GPU.
import torch

# Stop early rather than silently running a slow full training job on CPU.
if not torch.cuda.is_available():
    raise RuntimeError("No GPU detected. In Colab, select Runtime > Change runtime type > GPU, then rerun this cell.")

# Print the selected accelerator and PyTorch version for runtime diagnostics.
print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__)

# Import Colab's Drive helper and ask permission to access persistent storage.
from google.colab import drive
drive.mount("/content/drive")

# Use a Drive directory for checkpoints and the final model.
from pathlib import Path
OUTPUT_DIR = Path("/content/drive/MyDrive/dialogue-summarizer/pegasus-samsum")

# Create missing output folders before the trainer writes any checkpoints.
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Checkpoints and final model:", OUTPUT_DIR)

In [ ]:
# Import the Hugging Face Datasets loader.
from datasets import load_dataset

# Share model and length settings with the local project's defaults.
DATASET_NAME = "knkarthick/samsum"
MODEL_NAME = "google/pegasus-cnn_dailymail"
MAX_INPUT_LENGTH = 512
MAX_TARGET_LENGTH = 128

# Download or load the public SAMSum train, validation, and test splits.
dataset = load_dataset(DATASET_NAME)

# Verify all expected splits exist before starting model training.
required_splits = {"train", "validation", "test"}
if not required_splits.issubset(dataset.keys()):
    raise ValueError(f"Missing dataset splits: {required_splits - set(dataset.keys())}")

# Check every split has the conversation input and reference summary columns.
for split_name in required_splits:
    missing = {"dialogue", "summary"} - set(dataset[split_name].column_names)
    if missing:
        raise ValueError(f"Split {split_name} is missing columns: {missing}")

# Display split sizes so the loaded dataset can be checked at a glance.
print({name: len(split) for name, split in dataset.items()})

# Import auto classes that select Pegasus implementations from the model config.
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

# Load the model vocabulary and pretrained sequence-to-sequence weights.
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

# Convert a batch of text dialogues and summaries into model token IDs.
def tokenize_batch(batch):
    # Encode conversations as model inputs and truncate overly long examples.
    inputs = tokenizer(
        [text.strip() for text in batch["dialogue"]],
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
    )
    # Encode reference summaries separately because they are the training targets.
    targets = tokenizer(
        text_target=[text.strip() for text in batch["summary"]],
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
    )
    # The trainer compares generated tokens against these target token IDs.
    inputs["labels"] = targets["input_ids"]
    return inputs

# Tokenize each split in batches and discard original columns after conversion.
tokenized = {}
for split_name, split in dataset.items():
    tokenized[split_name] = split.map(
        tokenize_batch,
        batched=True,
        remove_columns=split.column_names,
        desc=f"Tokenizing {split_name}",
    )

In [ ]:
# Import a local ROUGE scorer and storage for summed scores.
from collections import defaultdict
from rouge_score.rouge_scorer import RougeScorer

# Measure unigram, bigram, longest-sequence, and summary-level word overlap.
rouge = RougeScorer(["rouge1", "rouge2", "rougeL", "rougeLsum"], use_stemmer=True)

# Trainer calls this function after generating summaries during validation.
def compute_metrics(eval_prediction):
    # Separate generated token IDs from reference label IDs.
    predictions, labels = eval_prediction
    # Some model outputs contain extra values; the first item is the token IDs.
    if isinstance(predictions, tuple):
        predictions = predictions[0]
    # Replace Trainer's ignored-label marker so the tokenizer can decode labels.
    labels = labels.copy()
    labels[labels == -100] = tokenizer.pad_token_id
    # Turn both generated and reference token IDs into readable strings.
    decoded_predictions = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    # Accumulate each ROUGE F1 score so we can calculate the batch average.
    totals = defaultdict(float)
    for prediction, reference in zip(decoded_predictions, decoded_labels):
        scores = rouge.score(reference.strip(), prediction.strip())
        for name, score in scores.items():
            totals[name] += score.fmeasure
    # Divide by the number of generated summaries and return each metric mean.
    count = max(len(decoded_predictions), 1)
    return {name: value / count for name, value in totals.items()}

In [ ]:
# Import the trainer, padding helper, and saved-checkpoint lookup.
from transformers import DataCollatorForSeq2Seq, Seq2SeqTrainer, Seq2SeqTrainingArguments
from transformers.trainer_utils import get_last_checkpoint

# Configure optimization, periodic checkpoint saving, and validation generation.
training_args = Seq2SeqTrainingArguments(
    # Save model and optimizer state in persistent Google Drive storage.
    output_dir=str(OUTPUT_DIR),
    # Run validation and save a recoverable checkpoint every 500 steps.
    eval_strategy="steps",
    save_strategy="steps",
    eval_steps=500,
    save_steps=500,
    logging_steps=100,
    # Set the optimizer's step size and the number of examples per device.
    learning_rate=5e-5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    weight_decay=0.01,
    warmup_ratio=0.1,
    # Process the full training split for three passes.
    num_train_epochs=3,
    # Generate summaries during evaluation, using the configured target limit.
    predict_with_generate=True,
    generation_max_length=MAX_TARGET_LENGTH,
    # Use half precision on the Colab GPU to reduce memory and speed up training.
    fp16=torch.cuda.is_available(),
    # Retain recent checkpoints and restore the best validation ROUGE-L model.
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="rougeL",
    greater_is_better=True,
    # Disable external experiment logging and fix the random seed.
    report_to="none",
    seed=42,
)

# Connect model, tokenized data, batch padding, and metric calculation.
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    # Pad shorter examples so all examples in a batch have compatible shapes.
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model),
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

# Find a saved checkpoint so an interrupted Colab session can resume.
last_checkpoint = get_last_checkpoint(str(OUTPUT_DIR))
print("Resuming from:", last_checkpoint or "starting a fresh run")
# Resume saved optimizer/model state when present; otherwise train from the base model.
train_result = trainer.train(resume_from_checkpoint=last_checkpoint)

# Save the final best checkpoint and its tokenizer for later inference.
trainer.save_model(str(OUTPUT_DIR))
tokenizer.save_pretrained(str(OUTPUT_DIR))
print(train_result.metrics)
print("Saved final model to:", OUTPUT_DIR)

## Evaluate and try a dialogue

The next cell evaluates on up to 100 held-out test examples. Then it summarizes the sample doctor-patient dialogue. ROUGE on 100 examples is a smoke check, not a full benchmark.

In [ ]:
# Import JSON formatting for readable printed test metrics.
import json

# Select at most 100 held-out examples for a quick test-split smoke check.
test_subset = tokenized["test"].select(range(min(100, len(tokenized["test"]))))
# Generate test summaries and print the resulting loss and ROUGE metrics.
print(json.dumps(trainer.predict(test_subset, metric_key_prefix="test").metrics, indent=2))

# Use the sample doctor-patient dialogue to try the fine-tuned model.
dialogue = """Doctor: What brings you here today?

Patient: I've had a headache and felt unusually tired for the last three days.

Doctor: Have you experienced a fever or vomiting?

Patient: No fever, but I haven't been sleeping well."""

# Encode the dialogue and place tensors on the same device as the model.
inputs = tokenizer(dialogue, return_tensors="pt", truncation=True, max_length=MAX_INPUT_LENGTH).to(model.device)
# Disable training-only behavior and gradient calculation during inference.
model.eval()
with torch.inference_mode():
    # Generate a bounded summary and discourage repeated n-gram phrases.
    generated = model.generate(
        **inputs,
        max_new_tokens=64,
        min_length=0,
        num_beams=4,
        no_repeat_ngram_size=3,
        length_penalty=1.0,
    )
# Decode generated tokens and replace Pegasus' visible newline markers.
summary = tokenizer.decode(generated[0], skip_special_tokens=True)
summary = " ".join(summary.replace("<n>", " ").split())
print("Summary:", summary)